# Jon2 AI — تدريب العقل العربي Qwen3.5-2B

الدفتر يجهّز بيئة GPU في Colab، يفحص ملفات التدريب وحقوقها، ثم يدرّب LoRA ويحفظ النتائج في Google Drive. اختر GPU من Runtime → Change runtime type.

**مهم:** شغّل خلايا التجهيز والفحص أولًا، وراجع تقرير الفحص قبل تشغيل خلية التدريب. التدريب الفعلي يغيّر أوزان adapter جديد ويحفظه في Drive، ولا يغيّر النموذج الأصلي ولا يفتح المحادثة العامة. الموافقات الحالية تسمح بالبحث التجريبي الخاص فقط.

In [ ]:
# 1) تنزيل نسخة العمل من فرع البناء
from pathlib import Path
import subprocess
repo_dir = Path('/content/jon2-ai')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', '--branch', 'feature/qwen25-arabic-lora-pipeline', '--single-branch', 'https://github.com/jonkabeel-star/jon2-ai.git', str(repo_dir)], check=True)
%cd /content/jon2-ai
print('Repository ready:', repo_dir)

In [ ]:
# 2) ربط Google Drive لحفظ الأوزان والنتائج بشكل دائم
from google.colab import drive
drive.mount('/content/drive')
output_dir = '/content/drive/MyDrive/Jon2_AI/jon2-qwen35-2b-lora'
Path('/content/drive/MyDrive/Jon2_AI').mkdir(parents=True, exist_ok=True)
print('Training output:', output_dir)

In [ ]:
# 3) التأكد من وجود GPU وتثبيت المكتبات
import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('ارجع إلى Runtime واختر GPU قبل المتابعة.')
print(torch.cuda.get_device_name(0))
subprocess.run(['python', '-m', 'pip', 'install', '-r', 'ai_engine/requirements-lora.txt'], check=True)

In [ ]:
# 4) فحص الحقوق والخصوصية وتقسيم البيانات — لا يغيّر أي أوزان
subprocess.run(['python', 'ai_engine/finetune_qwen_lora.py', '--dry-run'], check=True)

## قبل التدريب الفعلي
لا تنتقل للخلية التالية إلا إذا انتهى الفحص السابق بنجاح، وراجعت أن الملفات والمحادثات مناسبة. التدريب يستغرق وقتًا ويستهلك حصة GPU في Colab. لو الذاكرة غير كافية، أوقف التشغيل بدل حذف فحوص الحماية.

In [ ]:
# 5) التدريب الفعلي + تصدير GGUF Q4_K_M إلى Google Drive
# هذه الخلية فقط هي التي تنفذ optimizer steps وتنتج أوزانًا جديدة.
cmd = [
    'python', 'ai_engine/finetune_qwen_lora.py',
    '--execute', '--export-gguf',
    '--output', output_dir,
]
print('Starting real Qwen3.5-2B LoRA training...')
subprocess.run(cmd, check=True)
print('Training finished. Review jon2_lora_training_metadata.json and held-out results.')

In [ ]:
# 6) عرض الملفات الناتجة للتأكد من حفظها في Drive
for path in sorted(Path(output_dir).rglob('*')):
    if path.is_file():
        print(path, path.stat().st_size, 'bytes')